# Benchmark độ trễ: YOLOv8s và YOLOv8s-P2

Chạy **Run All** từ thư mục dự án (hoặc `scripts/`). Notebook đọc cùng 480 ảnh
test, hai checkpoint và ngưỡng đã khóa; không huấn luyện hoặc chọn tham số trên
test. Kết quả được ghi vào `evaluation/latency/` và `evaluation/figures/`.

B0/B2 dùng cùng YOLOv8s; B1/Proposed dùng cùng YOLOv8s-P2. Mỗi checkpoint
chỉ suy luận một lần trên mỗi ảnh/lượt; hai cấu hình của nó áp dụng ngưỡng riêng
lên cùng output sau NMS. Không có tiling nên `patches_per_image` là N/A.

In [1]:
import csv
import gc
import hashlib
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import Markdown, display
from PIL import Image
from ultralytics import YOLO

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/threshold_yolov8s.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án hoặc scripts/.")

SPLIT = ROOT / "data/splits/test.txt"
PROTOCOL = ROOT / "evaluation/B0_B1_yolov8s/week10_protocol_and_metrics.json"
QUALITY = ROOT / "evaluation/B0_B1_yolov8s/week10_primary_metrics.csv"
THRESHOLD = ROOT / "configs/threshold_yolov8s.yaml"
OUT = ROOT / "evaluation/latency"
FIGURES = ROOT / "evaluation/figures"

# Cấu hình đo (cố định trước khi xem kết quả). Cùng ảnh, thứ tự, batch và môi trường.
DEVICE = "mps" if torch.backends.mps.is_available() else "0" if torch.cuda.is_available() else "cpu"
IMGSZ = 1024
NMS_IOU = 0.7
PRED_CONF = 0.001
MAX_DET = 300
WARMUP_IMAGES = 12
REPEATS = 3
CRITICAL_IDS = {2, 3}  # Open_circuit, Short
CONFIGS = ("B0", "B1", "B2", "Proposed")

assert WARMUP_IMAGES > 0 and REPEATS >= 2

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1. Xác minh giao thức và dữ liệu đầu vào

Nếu hash test split/checkpoint khác thí nghiệm đã khóa, dừng trước khi đo.

In [2]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


protocol = json.loads(PROTOCOL.read_text(encoding="utf-8"))
threshold_cfg = yaml.safe_load(THRESHOLD.read_text(encoding="utf-8"))
quality = pd.read_csv(QUALITY).set_index("configuration").loc[list(CONFIGS)]
assert sha256(SPLIT) == protocol["test_split_sha256"]
assert protocol["imgsz"] == IMGSZ and protocol["nms_iou"] == NMS_IOU
assert threshold_cfg["selected_on"] == "validation" and threshold_cfg["test_not_used_for_threshold"] is True
assert list(quality.index) == list(CONFIGS)
images = [ROOT / line.strip() for line in SPLIT.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(images) == protocol["test_images"] == 480 and all(image.is_file() for image in images)
checkpoints = {}
for name in ("B0", "B1"):
    info = threshold_cfg["models"][name]
    checkpoint = ROOT / info["checkpoint"]
    assert checkpoint.is_file() and sha256(checkpoint) == info["checkpoint_sha256"] == protocol["checkpoint_sha256"][name]
    checkpoints[name] = checkpoint

thresholds = {
    "B0": (0.25, 0.25), "B1": (0.25, 0.25),
    "B2": (threshold_cfg["models"]["B0"]["tau_critical"],
           threshold_cfg["models"]["B0"]["tau_regular"]),
    "Proposed": (threshold_cfg["models"]["B1"]["tau_critical"],
                 threshold_cfg["models"]["B1"]["tau_regular"]),
}
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print("Test:", len(images), "images | device:", DEVICE, "| repeats:", REPEATS,
      "| checkpoint SHA256 verified")

Test: 480 images | device: mps | repeats: 3 | checkpoint SHA256 verified


## 2. Định nghĩa thời gian và chạy benchmark

- `model_only_ms`: trường `Results.speed['inference']` của Ultralytics, đoạn
  chạy mạng trên tensor đã chuẩn bị, **không** gồm đọc ảnh, preprocess hoặc NMS.
- `end_to_end_ms`: đọc+giải mã ảnh → `predict` (preprocess, network, NMS) →
  chuyển box sang CPU → lọc ngưỡng lớp và tạo danh sách kết quả.
- `pipeline_ms`: toàn bộ lời gọi `predict` (bao gồm overhead Python); không
  cộng `model_only_ms` vào `end_to_end_ms` lần nữa.

Đo mỗi ảnh với batch=1, đồng bộ GPU trước/sau đoạn đo, warm-up từng model.
Nạp checkpoint và warm-up nằm ngoài độ trễ end-to-end. MPS chỉ cho phép lấy
bộ nhớ ở thời điểm lấy mẫu, không phải đỉnh VRAM chính xác của toàn quá trình.

In [3]:
def synchronize():
    if DEVICE == "mps":
        torch.mps.synchronize()
    elif str(DEVICE).startswith("0") or str(DEVICE).startswith("cuda"):
        torch.cuda.synchronize()


def memory_bytes():
    if DEVICE == "mps" and hasattr(torch.mps, "driver_allocated_memory"):
        return int(torch.mps.driver_allocated_memory())
    if str(DEVICE).startswith("0") or str(DEVICE).startswith("cuda"):
        return int(torch.cuda.max_memory_allocated())
    return None


def measure_image(model, image, model_name, repeat, image_index):
    synchronize()
    start = time.perf_counter_ns()
    with Image.open(image) as source:
        pixels = np.asarray(source.convert("RGB"))
    decoded = time.perf_counter_ns()

    result = model.predict(source=pixels, imgsz=IMGSZ, batch=1,
                           device=DEVICE, conf=PRED_CONF, iou=NMS_IOU,
                           max_det=MAX_DET, verbose=False)[0]
    synchronize()
    predicted = time.perf_counter_ns()
    if result.boxes is None:
        classes = np.empty(0, dtype=np.int64)
        scores = np.empty(0, dtype=float)
        boxes = np.empty((0, 4), dtype=float)
    else:
        classes = result.boxes.cls.cpu().numpy().astype(np.int64)
        scores = result.boxes.conf.cpu().numpy()
        boxes = result.boxes.xyxy.cpu().numpy()
    synchronized = time.perf_counter_ns()

    # Both threshold strategies consume the same predictions (conf=.001, NMS=.7).
    rows = []
    selected_names = ("B0", "B2") if model_name == "B0" else ("B1", "Proposed")
    for name in selected_names:
        tau_critical, tau_regular = thresholds[name]
        t0 = time.perf_counter_ns()
        kept = np.flatnonzero(scores >= np.where(np.isin(classes, tuple(CRITICAL_IDS)),
                                                 tau_critical, tau_regular))
        detections = [{"class_id": int(classes[i]), "confidence": float(scores[i]),
                       "box_xyxy": boxes[i].tolist()} for i in kept]
        defective = bool(detections)
        t1 = time.perf_counter_ns()
        milliseconds = 1e-6
        rows.append({"configuration": name, "model": model_name, "repeat": repeat,
                     "image_index": image_index, "image": str(image.relative_to(ROOT)),
                     "read_decode_ms": (decoded-start)*milliseconds,
                     "pipeline_ms": (predicted-decoded)*milliseconds,
                     "cpu_transfer_ms": (synchronized-predicted)*milliseconds,
                     "threshold_and_output_ms": (t1-t0)*milliseconds,
                     "end_to_end_ms": (synchronized-start+t1-t0)*milliseconds,
                     "ultralytics_preprocess_ms": float(result.speed["preprocess"]),
                     "model_only_ms": float(result.speed["inference"]),
                     "ultralytics_postprocess_ms": float(result.speed["postprocess"]),
                     "detections": len(detections), "defective": defective,
                     "patches_per_image": "N/A"})
    return rows


def benchmark():
    measurements, resource_rows = [], []
    for model_name, checkpoint in checkpoints.items():
        gc.collect()
        if DEVICE == "mps":
            torch.mps.empty_cache()
        if str(DEVICE).startswith("0") or str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        model = YOLO(str(checkpoint))
        model_params = sum(p.numel() for p in model.model.parameters())
        print(f"{model_name}: warm-up {WARMUP_IMAGES} images; benchmark {REPEATS} × {len(images)}", flush=True)
        for image in images[:WARMUP_IMAGES]:
            with Image.open(image) as source:
                pixels = np.asarray(source.convert("RGB"))
            model.predict(source=pixels, imgsz=IMGSZ, batch=1, device=DEVICE,
                          conf=PRED_CONF, iou=NMS_IOU, max_det=MAX_DET, verbose=False)
        synchronize()
        sampled_memory = memory_bytes()
        for repeat in range(1, REPEATS+1):
            for index, image in enumerate(images):
                measurements.extend(measure_image(model, image, model_name, repeat, index))
                observed = memory_bytes()
                if observed is not None:
                    sampled_memory = max(sampled_memory or 0, observed)
            print(f"  {model_name}: pass {repeat}/{REPEATS} complete", flush=True)
        resource_rows.append({"model": model_name, "checkpoint_sha256": sha256(checkpoint),
                              "checkpoint_mb": checkpoint.stat().st_size/1e6,
                              "parameters": model_params, "memory_observed_mb":
                              sampled_memory/1e6 if sampled_memory is not None else "",
                              "memory_kind": "MPS driver allocation sampled, not peak" if DEVICE == "mps"
                              else "CUDA peak allocated" if DEVICE != "cpu" else "N/A"})
        del model
    return pd.DataFrame(measurements), pd.DataFrame(resource_rows)


per_image, resources = benchmark()
assert len(per_image) == len(images)*REPEATS*len(CONFIGS)
assert all(len(per_image[per_image.configuration == name]) == len(images)*REPEATS for name in CONFIGS)
assert (per_image.end_to_end_ms >= per_image.pipeline_ms).all()
per_image.to_csv(OUT / "per_image_latency.csv", index=False)
resources.to_csv(OUT / "model_resources.csv", index=False)

B0: warm-up 12 images; benchmark 3 × 480


  B0: pass 1/3 complete


  B0: pass 2/3 complete


  B0: pass 3/3 complete


B1: warm-up 12 images; benchmark 3 × 480


  B1: pass 1/3 complete


  B1: pass 2/3 complete


  B1: pass 3/3 complete


## 3. Tổng hợp thời gian và ghép với metric test đã khóa

P95 và median tính trên mọi ảnh gốc trong ba lượt đo. F2/Recall/mAP đọc từ
bảng chất lượng test gốc trên Kaggle CUDA; **không** thay bằng lần inference MPS.

In [4]:
def median_p95(values):
    return float(np.median(values)), float(np.percentile(values, 95))


summary = []
for name in CONFIGS:
    group = per_image[per_image.configuration == name]
    q = quality.loc[name]
    row = {"configuration": name, "model": q["model"],
           "samples": len(group), "unique_images": group.image.nunique(),
           "test_split_sha256": protocol["test_split_sha256"],
           "device": DEVICE, "batch": 1, "repeats": REPEATS,
           "warmup_images": WARMUP_IMAGES, "imgsz": IMGSZ,
           "inference_conf": PRED_CONF, "nms_iou": NMS_IOU,
           "tau_critical": q["tau_critical"], "tau_regular": q["tau_regular"],
           "F2_locked_cuda": q["F2"], "R_small_locked_cuda": q["R_small"],
           "critical_recall_locked_cuda": q["critical_recall"],
           "mAP50_95_locked_cuda": q["mAP50_95"], "patches_per_image": "N/A"}
    for metric in ("read_decode_ms", "pipeline_ms", "cpu_transfer_ms",
                   "threshold_and_output_ms", "end_to_end_ms", "model_only_ms",
                   "ultralytics_preprocess_ms", "ultralytics_postprocess_ms"):
        row[f"{metric}_median"], row[f"{metric}_p95"] = median_p95(group[metric])
    resource = resources.set_index("model").loc[q["model"]]
    row.update({"checkpoint_mb": resource.checkpoint_mb,
                "parameters": int(resource.parameters),
                "memory_observed_mb": resource.memory_observed_mb,
                "memory_kind": resource.memory_kind})
    summary.append(row)

summary = pd.DataFrame(summary)
summary.to_csv(OUT / "latency_summary.csv", index=False)
assert len(summary) == 4 and summary.samples.eq(480*REPEATS).all()
display(summary[["configuration", "end_to_end_ms_median", "end_to_end_ms_p95",
                 "model_only_ms_median", "model_only_ms_p95",
                 "F2_locked_cuda", "R_small_locked_cuda", "memory_observed_mb"]].round(4))

,configuration,end_to_end_ms_median,end_to_end_ms_p95,model_only_ms_median,model_only_ms_p95,F2_locked_cuda,R_small_locked_cuda,memory_observed_mb
0,B0,85.0979,116.5577,40.2104,50.6530,0.8925,0.9548,1184.4977
1,B1,99.9692,138.0444,49.9733,68.6225,0.8831,0.9289,1150.9432
2,B2,85.0153,116.4928,40.2104,50.6530,0.8486,0.9329,1184.4977
3,Proposed,99.8898,137.9720,49.9733,68.6225,0.8449,0.9087,1150.9432


## 4. Biểu đồ và kết luận trade-off

In [5]:
COLORS = {"B0": "#2563eb", "B1": "#f97316", "B2": "#16a34a", "Proposed": "#9333ea"}


def tradeoff_plot(y_col, ylabel, filename):
    fig, ax = plt.subplots(figsize=(9, 6))
    for _, row in summary.iterrows():
        name = row.configuration
        x = row.end_to_end_ms_median
        y = row[y_col]
        ax.scatter(x, y, s=110, color=COLORS[name], zorder=3)
        ax.annotate(name, (x, y), xytext=(7, 7), textcoords="offset points", fontsize=11)
        ax.plot([x, row.end_to_end_ms_p95], [y, y], color=COLORS[name], alpha=0.4, lw=2)
    ax.set(xlabel="End-to-end latency (ms/ảnh, median; vạch tới p95, MPS)",
           ylabel=ylabel + " (locked Kaggle CUDA test)",
           title="YOLOv8s vs YOLOv8s-P2 — chất lượng / thời gian")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=180)
    plt.close(fig)


tradeoff_plot("F2_locked_cuda", "F2", "f2_vs_latency.png")
tradeoff_plot("R_small_locked_cuda", "Recall small", "small_recall_vs_latency.png")


def table_line(r):
    return (f"| {r.configuration} | {r.model} | {r.F2_locked_cuda:.4f} | "
            f"{r.R_small_locked_cuda:.4f} | {r.critical_recall_locked_cuda:.4f} | "
            f"{r.mAP50_95_locked_cuda:.4f} | {r.end_to_end_ms_median:.2f} / "
            f"{r.end_to_end_ms_p95:.2f} | {r.model_only_ms_median:.2f} / "
            f"{r.model_only_ms_p95:.2f} | {r.memory_observed_mb:.1f} |")


rows = {r.configuration: r for _, r in summary.iterrows()}
speed_ratio = rows["B1"].end_to_end_ms_median / rows["B0"].end_to_end_ms_median
lines = ["# Chất lượng–tốc độ: YOLOv8s và YOLOv8s-P2", "",
         "## Giao thức đo", "",
         f"- Đo lúc {datetime.now(timezone.utc).isoformat()} trên {platform.platform()} "
         f"({platform.processor() or platform.machine()}); PyTorch {torch.__version__}, "
         f"Ultralytics {ultralytics.__version__}, device `{DEVICE}`.",
         f"- Test: {len(images)} ảnh gốc, hash `{protocol['test_split_sha256']}`; "
         f"imgsz={IMGSZ}, batch=1, conf suy luận={PRED_CONF}, NMS IoU={NMS_IOU}, "
         f"max_det={MAX_DET}; warm-up={WARMUP_IMAGES}, đo {REPEATS} lượt/ảnh/checkpoint.",
         "- Đồng bộ thiết bị trước/sau `predict`; end-to-end bao gồm đọc/giải mã ảnh, "
         "predict (preprocess, forward, NMS), chuyển box sang CPU và lọc confidence + "
         "tạo danh sách kết quả. Không gồm nạp model và warm-up, không gồm ghi CSV/hình.",
         "- Model-only lấy `Results.speed['inference']` của Ultralytics (batch=1), "
         "không cộng vào end-to-end lần thứ hai. Median/p95 tính trên 1.440 mẫu ảnh "
         "mỗi cấu hình (480 × 3). B0/B2 và B1/Proposed dùng chung suy luận nhưng "
         "lọc threshold được bấm giờ độc lập.",
         "- Cột bộ nhớ trên MPS là mức driver allocation cao nhất *được lấy mẫu* "
         "trong khi chạy từng checkpoint; không phải GPU peak tuyệt đối.",
         "- Chất lượng (F2, Recall, mAP) giữ nguyên từ "
         "`evaluation/B0_B1_yolov8s/week10_primary_metrics.csv` chạy trên Kaggle CUDA. "
         "Latency đo trên MPS, nên đây là biểu đồ kết hợp hai môi trường; lần tái "
         "chạy tuần trước trên MPS có một số sai khác TP/FP/FN. Không diễn giải biểu "
         "đồ thành đánh giá chất lượng và latency đồng thời trên cùng GPU.", "",
         "## Bảng tổng hợp", "",
         "| Cấu hình | Model | F2 | R-small | R-critical | mAP50-95 | "
         "End-to-end median/p95 (ms) | Model-only median/p95 (ms) | MPS allocation mẫu (MB) |",
         "|---|---|---:|---:|---:|---:|---:|---:|---:|"]
lines.extend(table_line(row) for _, row in summary.iterrows())
lines += ["", f"- Trên MPS, tỷ số latency end-to-end median B1/B0 = "
          f"**{speed_ratio:.2f}×** (B0 {rows['B0'].end_to_end_ms_median:.2f} ms; "
          f"B1 {rows['B1'].end_to_end_ms_median:.2f} ms). "
          "P2 không cải thiện Recall small hoặc F2 trên test gốc đã khóa; "
          "đánh đổi tăng/giảm latency được mô tả bằng số đo trên phần cứng này.",
          "- B2/Proposed chỉ đổi ngưỡng hậu xử lý; thời gian model-only của mỗi "
          "cặp B0/B2 và B1/Proposed là cùng một lần suy luận. Chênh lệch end-to-end "
          "nếu có do lượng output và lọc/ngữ cảnh đo, không phải một mạng khác.",
          "- Không áp dụng số patch/ảnh: P2 là thay đổi kiến trúc detection head, "
          "không phải SAHI/tiling. Không suy ra FPS thời gian thực nếu chưa đặt ngưỡng ứng dụng.",
          "", "## Biểu đồ và dữ liệu gốc", "",
          "- [F2 so với latency](../figures/f2_vs_latency.png)",
          "- [Recall small so với latency](../figures/small_recall_vs_latency.png)",
          "- `per_image_latency.csv`: các thành phần thời gian theo ảnh/lượt/cấu hình.",
          "- `latency_summary.csv`: median/p95 và metric test đã khóa.",
          "- `model_resources.csv`: checkpoint, tham số và bộ nhớ lấy mẫu.",
          "- Tái chạy: mở `scripts/19_benchmark_inference_latency.ipynb` và chọn Run All "
          "trong Jupyter. Không thay ngưỡng hay test split.", ""]
(OUT / "quality_speed_report.md").write_text("\n".join(lines), encoding="utf-8")
display(Markdown("\n".join(lines)))

# Chất lượng–tốc độ: YOLOv8s và YOLOv8s-P2

## Giao thức đo

- Đo lúc 2026-10-26T03:38:13.279483+00:00 trên macOS-15.7-arm64-arm-64bit-Mach-O (arm); PyTorch 2.14.0, Ultralytics 8.4.153, device `mps`.
- Test: 480 ảnh gốc, hash `488eed2bd670c39988c741d0214df47ef9b5c4f3ff442eb4259c6f3b7163e459`; imgsz=1024, batch=1, conf suy luận=0.001, NMS IoU=0.7, max_det=300; warm-up=12, đo 3 lượt/ảnh/checkpoint.
- Đồng bộ thiết bị trước/sau `predict`; end-to-end bao gồm đọc/giải mã ảnh, predict (preprocess, forward, NMS), chuyển box sang CPU và lọc confidence + tạo danh sách kết quả. Không gồm nạp model và warm-up, không gồm ghi CSV/hình.
- Model-only lấy `Results.speed['inference']` của Ultralytics (batch=1), không cộng vào end-to-end lần thứ hai. Median/p95 tính trên 1.440 mẫu ảnh mỗi cấu hình (480 × 3). B0/B2 và B1/Proposed dùng chung suy luận nhưng lọc threshold được bấm giờ độc lập.
- Cột bộ nhớ trên MPS là mức driver allocation cao nhất *được lấy mẫu* trong khi chạy từng checkpoint; không phải GPU peak tuyệt đối.
- Chất lượng (F2, Recall, mAP) giữ nguyên từ `evaluation/B0_B1_yolov8s/week10_primary_metrics.csv` chạy trên Kaggle CUDA. Latency đo trên MPS, nên đây là biểu đồ kết hợp hai môi trường; lần tái chạy tuần trước trên MPS có một số sai khác TP/FP/FN. Không diễn giải biểu đồ thành đánh giá chất lượng và latency đồng thời trên cùng GPU.

## Bảng tổng hợp

| Cấu hình | Model | F2 | R-small | R-critical | mAP50-95 | End-to-end median/p95 (ms) | Model-only median/p95 (ms) | MPS allocation mẫu (MB) |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| B0 | B0 | 0.8925 | 0.9548 | 0.9604 | 0.4222 | 85.10 / 116.56 | 40.21 / 50.65 | 1184.5 |
| B1 | B1 | 0.8831 | 0.9289 | 0.9406 | 0.3999 | 99.97 / 138.04 | 49.97 / 68.62 | 1150.9 |
| B2 | B0 | 0.8486 | 0.9329 | 0.9765 | 0.4222 | 85.02 / 116.49 | 40.21 / 50.65 | 1184.5 |
| Proposed | B1 | 0.8449 | 0.9087 | 0.9567 | 0.3999 | 99.89 / 137.97 | 49.97 / 68.62 | 1150.9 |

- Trên MPS, tỷ số latency end-to-end median B1/B0 = **1.17×** (B0 85.10 ms; B1 99.97 ms). P2 không cải thiện Recall small hoặc F2 trên test gốc đã khóa; đánh đổi tăng/giảm latency được mô tả bằng số đo trên phần cứng này.
- B2/Proposed chỉ đổi ngưỡng hậu xử lý; thời gian model-only của mỗi cặp B0/B2 và B1/Proposed là cùng một lần suy luận. Chênh lệch end-to-end nếu có do lượng output và lọc/ngữ cảnh đo, không phải một mạng khác.
- Không áp dụng số patch/ảnh: P2 là thay đổi kiến trúc detection head, không phải SAHI/tiling. Không suy ra FPS thời gian thực nếu chưa đặt ngưỡng ứng dụng.

## Biểu đồ và dữ liệu gốc

- [F2 so với latency](../figures/f2_vs_latency.png)
- [Recall small so với latency](../figures/small_recall_vs_latency.png)
- `per_image_latency.csv`: các thành phần thời gian theo ảnh/lượt/cấu hình.
- `latency_summary.csv`: median/p95 và metric test đã khóa.
- `model_resources.csv`: checkpoint, tham số và bộ nhớ lấy mẫu.
- Tái chạy: mở `scripts/19_benchmark_inference_latency.ipynb` và chọn Run All trong Jupyter. Không thay ngưỡng hay test split.
